# DRACO-Lite: Client → URL4 Engine → AI Gateway

This notebook exercises the complete local vertical slice:

1. The lazy Client discovers the Engine's Models and Benchmarks.
2. `Client.evaluate(...)` fetches and verifies the DRACO-Lite YAML manifest and constructs one
   flat Candidate URL4 before spending.
3. The Client uses the url4-cloud token and WebSocket lifecycle.
4. The configured URL4 node loads one pinned DRACO case, calls AI Gateway for the answer, runs ten
   rubric judges, aggregates the grades, and returns an `sf.Report`.

> **Cost warning:** the evaluation cell performs one answer call and ten judge calls. Discovery
> makes no model calls.

## Before running

The local AI Gateway must be running on `127.0.0.1:9105`, and the isolated Engine demo must be
running on `127.0.0.1:9108`. See `apps/url4-cloud/DRACO_LITE_DEMO.md` in the Engine demo branch for
the exact commands.

In [ ]:
import screamingface as sf

## Discover the local Engine

These catalogue calls are read-only and make no model requests.

In [ ]:
sf.benchmarks.list()

In [ ]:
sf.models.list()

## Define one Candidate

The demo uses the provider-prefixed Anthropic route currently accepted by the local AI Gateway.

In [ ]:
ANSWER_INSTRUCTIONS = """Answer the research question completely.
Compare the estimators and their assumptions precisely, address pre-trend testing, and cite
specific papers and evidence where useful."""

candidate = sf.Model(
    "anthropic/claude-haiku-4-5",
    instructions=ANSWER_INSTRUCTIONS,
    max_output_tokens=4096,
)

candidate

## Evaluate the benchmark

Running the next cell makes **11 model calls**: one Candidate answer and ten concurrent
rubric-judge calls. Manifest verification and URL4 compilation happen first; the Client starts no
Candidate Run if that validation fails.

In [ ]:
events = []


def on_event(event: sf.Event) -> None:
    events.append(event)
    print(f"{event.sequence:02d} {event.kind}")


with sf.Client() as client:
    report = client.evaluate(
        candidate,
        benchmark="draco-lite",
        limit=1,
        on_event=on_event,
    )
report

In [ ]:
print(report.candidates.only.url4)

In [ ]:
report.candidates.only.metrics

In [ ]:
report.usage

In [ ]:
print(report.to_json())